In [ ]:
!pip install opencv-python-headless

In [ ]:
import cv2
import matplotlib.pyplot as plt
import os

video_file = os.path.join(VIDEO_PATH, f"{video_id}.mp4")

cap = cv2.VideoCapture(video_file)

if not cap.isOpened():
    print("Error: Could not open video.")
else:
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print("Video ID:", video_id)
    print("Total frames:", total_frames)
    print("FPS:", fps)
    print("Resolution:", width, "x", height)

    success, frame = cap.read()

    if success:
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        plt.figure(figsize=(8, 5))
        plt.imshow(frame_rgb)
        plt.title("First Frame of ASL Video")
        plt.axis("off")
        plt.show()
    else:
        print("Could not read the first frame.")

    cap.release()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2

cap = cv2.VideoCapture(video_file)

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

if total_frames == 0:
    print("No frames available in this video.")
else:
    frame_positions = np.linspace(
        0,
        total_frames - 1,
        min(6, total_frames),
        dtype=int
    )

    frames = []

    for position in frame_positions:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(position))
        success, frame = cap.read()

        if success:
            frame = cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2RGB
            )
            frames.append((position, frame))

    cap.release()

    fig, axes = plt.subplots(
        1,
        len(frames),
        figsize=(18, 4)
    )

    if len(frames) == 1:
        axes = [axes]

    for ax, (position, frame) in zip(axes, frames):
        ax.imshow(frame)
        ax.set_title(f"Frame {position}")
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    print("Frames successfully extracted:", len(frames))

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

cap = cv2.VideoCapture(video_file)

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

frame_positions = np.linspace(
    0,
    total_frames - 1,
    min(8, total_frames),
    dtype=int
)

processed_frames = []

for position in frame_positions:
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(position))
    success, frame = cap.read()

    if success:
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        resized_frame = cv2.resize(
            frame,
            (224, 224)
        )

        normalized_frame = resized_frame.astype(
            np.float32
        ) / 255.0

        processed_frames.append(normalized_frame)

cap.release()

processed_frames = np.array(processed_frames)

print("Processed video shape:", processed_frames.shape)
print("Data type:", processed_frames.dtype)
print("Minimum pixel value:", processed_frames.min())
print("Maximum pixel value:", processed_frames.max())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

print("Processed frames shape:", processed_frames.shape)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))

for i, ax in enumerate(axes.flat):
    if i < len(processed_frames):
        ax.imshow(processed_frames[i])
        ax.set_title(f"Frame {i + 1}")
    ax.axis("off")

plt.suptitle("ASL Video: Processed Frames")
plt.tight_layout()
plt.show()

In [ ]:
import cv2
import numpy as np

def preprocess_video(video_path, num_frames=8, image_size=224):

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print("Could not open video:", video_path)
        return None

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames <= 0:
        cap.release()
        print("Video contains no readable frames.")
        return None

    positions = np.linspace(
        0,
        total_frames - 1,
        min(num_frames, total_frames),
        dtype=int
    )

    frames = []

    for position in positions:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(position))
        success, frame = cap.read()

        if not success:
            continue

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        frame = cv2.resize(
            frame,
            (image_size, image_size)
        )

        frame = frame.astype(np.float32) / 255.0

        frames.append(frame)

    cap.release()

    if len(frames) == 0:
        return None

    return np.array(frames, dtype=np.float32)


sample_frames = preprocess_video(video_file)

if sample_frames is not None:
    print("Preprocessing successful!")
    print("Output shape:", sample_frames.shape)
    print("Data type:", sample_frames.dtype)

In [ ]:
import os
import pandas as pd
import numpy as np

sample_records = video_df[
    video_df["url"].astype(str).str.startswith("http")
].head(5)

batch_results = []

for _, record in sample_records.iterrows():

    current_id = record["video_id"]

    current_path = os.path.join(
        VIDEO_PATH,
        f"{current_id}.mp4"
    )

    if not os.path.exists(current_path):
        print("Video not downloaded:", current_id)
        continue

    frames = preprocess_video(current_path)

    if frames is not None:
        batch_results.append({
            "video_id": current_id,
            "gloss": record["gloss"],
            "signer_id": record["signer_id"],
            "frames": frames
        })

print("\nBatch processing complete.")
print("Videos successfully processed:", len(batch_results))

for result in batch_results:
    print(
        "Video:", result["video_id"],
        "| Sign:", result["gloss"],
        "| Shape:", result["frames"].shape
    )

In [ ]:
import os
import requests
import pandas as pd

os.makedirs(VIDEO_PATH, exist_ok=True)

download_results = []

for _, record in video_df.head(5).iterrows():

    video_id = str(record["video_id"])
    video_path = os.path.join(VIDEO_PATH, f"{video_id}.mp4")
    url = str(record["url"]).strip()

    if url.startswith("[") and "](" in url:
        url = url.split("](", 1)[1].rstrip(")")

    if os.path.exists(video_path):
        print("Already downloaded:", video_id)
        download_results.append({
            "video_id": video_id,
            "status": "Already available"
        })
        continue

    try:
        response = requests.get(
            url,
            timeout=30,
            allow_redirects=True
        )

        content_type = response.headers.get(
            "Content-Type", ""
        ).lower()

        if response.status_code == 200 and "video/" in content_type:
            with open(video_path, "wb") as file:
                file.write(response.content)

            print("Downloaded:", video_id)

            download_results.append({
                "video_id": video_id,
                "status": "Downloaded"
            })
        else:
            print("Unavailable or unexpected response:", video_id)

            download_results.append({
                "video_id": video_id,
                "status": f"HTTP {response.status_code}; {content_type}"
            })

        response.close()

    except requests.RequestException as error:
        print("Failed:", video_id, "-", error)

        download_results.append({
            "video_id": video_id,
            "status": str(error)
        })

print("\nDownload summary:")
display(pd.DataFrame(download_results))

In [ ]:
import pandas as pd
from urllib.parse import urlparse

url_check = video_df.head(20).copy()

url_check["url"] = (
    url_check["url"]
    .astype(str)
    .str.strip()
    .str.replace(r"^\[", "", regex=True)
    .str.replace(r"\]\((https?://.*)\)$", r"\1", regex=True)
)

url_check["domain"] = url_check["url"].apply(
    lambda url: urlparse(url).netloc
)

url_check["looks_like_mp4"] = url_check["url"].str.lower().str.contains(
    r"\.mp4(?:$|\?)",
    regex=True,
    na=False
)

print("Video source distribution:")
display(
    url_check["domain"].value_counts().rename_axis("domain").reset_index(name="records")
)

print("\nURLs that appear to point directly to MP4 files:")
display(
    url_check[
        ["video_id", "gloss", "source", "domain", "looks_like_mp4", "url"]
    ]
)

In [ ]:
!git clone https://github.com/dxli94/WLASL.git /content/WLASL

In [ ]:
import os

START_KIT_PATH = "/content/WLASL/start_kit"

print("Start kit exists:", os.path.exists(START_KIT_PATH))

print("\nAvailable files:")

for filename in os.listdir(START_KIT_PATH):
    print(filename)

In [ ]:
DOWNLOADER_PATH = "/content/WLASL/start_kit/video_downloader.py"

with open(DOWNLOADER_PATH, "r", encoding="utf-8") as file:
    downloader_code = file.readlines()

print("Downloader script length:", len(downloader_code), "lines")
print("\nFirst 180 lines:\n")

for line_number, line in enumerate(downloader_code[:180], start=1):
    print(f"{line_number:03}: {line}", end="")

In [ ]:
!pip install -q yt-dlp

In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "yt_dlp", "--version"],
    capture_output=True,
    text=True
)

print("yt-dlp available:", result.returncode == 0)
print("Version:", result.stdout.strip())

if result.returncode != 0:
    print(result.stderr)

In [ ]:
import os
import subprocess
import sys

YOUTUBE_TEST_ID = "68011"
YOUTUBE_TEST_URL = "https://www.youtube.com/watch?v=0UsjUE-TXns"

os.makedirs(VIDEO_PATH, exist_ok=True)

result = subprocess.run(
    [
        sys.executable, "-m", "yt_dlp",
        "--no-playlist",
        "--max-filesize", "100M",
        "-f", "mp4/best",
        "-o", os.path.join(VIDEO_PATH, "%(id)s.%(ext)s"),
        YOUTUBE_TEST_URL
    ],
    capture_output=True,
    text=True,
    timeout=180
)

print("Return code:", result.returncode)
print("\nDownload log:\n")
print(result.stdout[-5000:])

if result.returncode != 0:
    print("\nError details:\n")
    print(result.stderr[-3000:])

In [ ]:
print("Files in video folder:")

for filename in os.listdir(VIDEO_PATH):
    path = os.path.join(VIDEO_PATH, filename)
    size_mb = os.path.getsize(path) / (1024 * 1024)

    print(f"{filename} — {size_mb:.2f} MB")

In [ ]:
import shutil

print("Node.js path:", shutil.which("node"))
print("Deno path:", shutil.which("deno"))

In [ ]:
import os
import subprocess
import sys

YOUTUBE_URL = "https://www.youtube.com/watch?v=0UsjUE-TXns"

result = subprocess.run(
    [
        sys.executable, "-m", "yt_dlp",
        "--js-runtimes", "node",
        "--no-playlist",
        "--max-filesize", "100M",
        "-f", "mp4/best",
        "-o", os.path.join(VIDEO_PATH, "%(id)s.%(ext)s"),
        YOUTUBE_URL
    ],
    capture_output=True,
    text=True,
    timeout=180
)

print("Return code:", result.returncode)
print("\nOutput:")
print(result.stdout[-3000:])

if result.returncode != 0:
    print("\nError:")
    print(result.stderr[-3000:])

In [ ]:
import subprocess
import sys

print("Node.js version:")
subprocess.run(["/tools/node/bin/node", "--version"])

print("\nUpdating yt-dlp with its default dependencies...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-U", "yt-dlp[default]"
], check=True)

print("\nUpdated yt-dlp version:")
subprocess.run([sys.executable, "-m", "yt_dlp", "--version"])

In [ ]:
result = subprocess.run(
    [
        sys.executable, "-m", "yt_dlp",
        "--js-runtimes", "node:/tools/node/bin/node",
        "--remote-components", "ejs:github",
        "--no-playlist",
        "--list-formats",
        "https://www.youtube.com/watch?v=0UsjUE-TXns"
    ],
    capture_output=True,
    text=True,
    timeout=120
)

print("Return code:", result.returncode)
print(result.stdout[-3000:])
print(result.stderr[-3000:])


In [ ]:
import os
import subprocess
import sys

video_url = "https://www.youtube.com/watch?v=0UsjUE-TXns"
output_template = os.path.join(VIDEO_PATH, "%(id)s.%(ext)s")

result = subprocess.run(
    [
        sys.executable, "-m", "yt_dlp",
        "--js-runtimes", "node:/tools/node/bin/node",
        "--remote-components", "ejs:github",
        "--no-playlist",
        "-f", "135",
        "-o", output_template,
        video_url
    ],
    capture_output=True,
    text=True,
    timeout=180
)

print("Return code:", result.returncode)
print(result.stdout[-3000:])

if result.returncode != 0:
    print(result.stderr[-3000:])

In [ ]:
from urllib.parse import urlparse
import pandas as pd

url_report = video_df.copy()

url_report["domain"] = url_report["url"].apply(
    lambda x: urlparse(str(x).strip()).netloc.lower()
)

print("Total annotation records:", len(url_report))
print("\nRecords by URL domain:")
print(url_report["domain"].value_counts().head(15))

print("\nSample non-YouTube URLs:")
non_youtube = url_report[
    ~url_report["domain"].str.contains("youtube.com|youtu.be", na=False)
]

print(
    non_youtube[
        ["gloss", "video_id", "signer_id", "url", "domain"]
    ].head(10).to_string(index=False)
)

In [ ]:
import re
import requests
from urllib.parse import urlparse

def clean_url(value):
    match = re.search(r"\]\((https?://[^)]+)\)", str(value))
    if match:
        return match.group(1)
    return str(value).strip()

test_df = video_df.copy()
test_df["clean_url"] = test_df["url"].apply(clean_url)
test_df["domain"] = test_df["clean_url"].apply(
    lambda x: urlparse(x).netloc.lower()
)

# Prefer direct MP4 links from SignStock
samples = test_df[
    (test_df["domain"] == "signstock.blob.core.windows.net")
    & test_df["clean_url"].str.lower().str.endswith(".mp4")
].drop_duplicates(subset=["video_id"]).head(5)

for _, row in samples.iterrows():
    try:
        response = requests.get(
            row["clean_url"],
            headers={"Range": "bytes=0-1023"},
            timeout=20,
            stream=True
        )

        print(
            f"ID: {row['video_id']} | "
            f"Gloss: {row['gloss']} | "
            f"Signer: {row['signer_id']}"
        )
        print("Status:", response.status_code)
        print("Content-Type:", response.headers.get("Content-Type"))
        print()

        response.close()

    except requests.RequestException as e:
        print(f"ID: {row['video_id']} | Request failed: {e}\n")

In [ ]:
import os
import requests

os.makedirs(VIDEO_PATH, exist_ok=True)

download_results = []

for _, row in samples.iterrows():
    video_id = str(row["video_id"]).zfill(5)
    video_url = row["clean_url"]
    save_path = os.path.join(VIDEO_PATH, f"{video_id}.mp4")

    try:
        response = requests.get(video_url, stream=True, timeout=60)
        response.raise_for_status()

        with open(save_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

        size_mb = os.path.getsize(save_path) / (1024 * 1024)

        download_results.append({
            "video_id": video_id,
            "gloss": row["gloss"],
            "signer_id": row["signer_id"],
            "size_mb": round(size_mb, 2),
            "status": "Downloaded" if size_mb > 0 else "Empty file"
        })

    except Exception as e:
        download_results.append({
            "video_id": video_id,
            "gloss": row["gloss"],
            "signer_id": row["signer_id"],
            "size_mb": 0,
            "status": str(e)[:100]
        })

print(pd.DataFrame(download_results).to_string(index=False))

In [ ]:
import os
import cv2
import pandas as pd

validation_results = []

for _, row in samples.iterrows():
    video_id = str(row["video_id"]).zfill(5)
    video_path = os.path.join(VIDEO_PATH, f"{video_id}.mp4")

    cap = cv2.VideoCapture(video_path)

    opened = cap.isOpened()
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    success, frame = cap.read()
    frame_readable = success and frame is not None

    cap.release()

    validation_results.append({
        "video_id": video_id,
        "gloss": row["gloss"],
        "signer_id": row["signer_id"],
        "opened": opened,
        "frame_count": frame_count,
        "fps": round(fps, 2),
        "frame_readable": frame_readable
    })

validation_df = pd.DataFrame(validation_results)
print(validation_df.to_string(index=False))